# Deploy Tool — Functional Requirements Document

**Project**: Deploy Tool  
**Date**: 2026-06-26  
**Status**: Draft v2 — Updated with review feedback  

---

## 1. Overview

The Deploy Tool is a web-based application that provides a UI for deploying changes from two GitHub repositories — **nbn-daemon** and **Unity** — to target filer machines (identified by IP address).

The tool allows developers to:
1. Select a repository (`nbn-daemon` or `unity`)
2. Specify a Git branch to deploy (via searchable dropdown)
3. Specify a target filer IP address
4. Trigger the deployment and monitor real-time logs in the UI
5. **Bookmark** the deployment configuration (repo, branch, filer IP) via URL query parameters for quick re-deployment

## 2. Tech Stack

| Layer | Technology |
|-------|------------|
| **Backend** | Python — FastAPI |
| **Frontend** | React + TypeScript + Tailwind CSS |
| **Deployment Execution** | Subprocess calls to existing deploy scripts in cloned repos |
| **Real-time Logs** | WebSocket (FastAPI → React) |
| **Source Repos** | Both via HTTPS + PAT: `https://github.com/nasuni/nbn-daemon.git`, `https://github.com/nasuni/unity.git` |

## 3. Target Repositories

### 3.1 nbn-daemon

- **GitHub URL**: `https://github.com/nasuni/nbn-daemon.git` (HTTPS + PAT)
- **Deploy mechanism**: `make deploy-rpm FILER=<filerIP>` (runs from repo root)
- **Build step**: The `deploy-rpm` target internally calls `build_rpm()` (which runs `make build-rpm`), so a separate build step is **not** needed.
- **Under the hood**: `PYTHONPATH=. uv run rpm/deploy/deploy.py <filerIP>` → builds RPM via Docker → SCPs it to filer → installs via SSH
- **Prerequisites**: Docker must be installed and running on the host machine

### 3.2 Unity

- **GitHub URL**: `https://github.com/nasuni/unity.git` (HTTPS + PAT)
- **Deploy mechanism**: `python/tools/sync-dev.py` script + post-sync cleanup & service restarts
- **Usage**: `python python/tools/sync-dev.py --restart <filerIP>`
- **Post-sync steps** (required after sync-dev.py completes):
  1. Delete stale bytecode: `find /opt/nasuni/lib/nasuni/ -name '*.pyc' -delete && find /opt/nasuni/lib/nasuni/ -name '*.pyo' -delete`
  2. Restart services: `systemctl restart filer-route-http qman && systemctl restart httpd`
- **Full post-sync one-liner** (executed via SSH on filer):
  ```bash
  ssh -p222 root@<FILER_IP> "find /opt/nasuni/lib/nasuni/ -name '*.pyc' -delete && find /opt/nasuni/lib/nasuni/ -name '*.pyo' -delete && systemctl restart filer-route-http qman && systemctl restart httpd"
  ```
- **Prerequisites**: SSH access (port 222) to the target filer from the host machine

### 3.2.1 Unity Post-Sync Details

**Why bytecode deletion is needed:**  
Python 2 uses cached `.pyc` files. If they're newer than the `.py` source, the old compiled code runs instead of your changes.

**Services that must be restarted:**

| Service | Command | What it does |
|---------|---------|---------------|
| `filer-route-http` | `systemctl restart filer-route-http` | Standalone HTTP daemon handling NMC→filer route calls (port 8765). Long-lived, caches all modules at startup. |
| `qman` | `systemctl restart qman` | Queue manager handling NMC commands via SQS/FIFO. Also imports route modules. |
| `httpd` | `systemctl restart httpd` | Apache with WSGI workers serving the filer admin UI. **Must be `restart`, not `reload`** — reload keeps old workers alive with stale in-memory modules. |

**What `sync-dev.py --restart` does NOT handle:**
- It syncs source files but does **NOT** delete bytecode
- It does **NOT** restart `filer-route-http` (only httpd/taskman via `--restart` flag)
- The `--restart` flag does `reload httpd` + `restart taskman` — but that's insufficient for route/credential changes

**Quick reference — when to restart what:**

| Changed code in | Restart |
|-----------------|---------|
| filer_shims | `filer-route-http` + `qman` + `httpd` |
| routes | `filer-route-http` + `qman` + `httpd` |
| nasuni | `filer-route-http` + `qman` + `httpd` |
| gui (filer UI) | `httpd` only |
| nmc_ui (NMC UI) | `httpd` on the **NMC**, not filer |

**Decision**: For v1, the deploy tool will always perform the full restart (`filer-route-http` + `qman` + `httpd`) after every Unity sync, regardless of which files changed. This is safe and ensures changes always take effect.

## 4. System Architecture

```
┌──────────────────────────────────────────────────────────┐
│                    Developer Browser                      │
│                                                          │
│  React + TypeScript + Tailwind CSS Frontend              │
│  ┌────────────────────────────────────────────────────┐  │
│  │  - Repo selector (nbn-daemon / unity)              │  │
│  │  - Branch searchable dropdown                      │  │
│  │  - Filer IP input                                  │  │
│  │  - Deploy button                                   │  │
│  │  - Real-time log viewer (WebSocket)                │  │
│  │  - URL reflects repo + branch + filerIP (bookmark) │  │
│  └────────────────────────────────────────────────────┘  │
└──────────────────────┬───────────────────────────────────┘
                       │ HTTP + WebSocket
┌──────────────────────▼───────────────────────────────────┐
│              FastAPI Backend (Python)                      │
│  ┌────────────────────────────────────────────────────┐  │
│  │  POST /api/deploy    — trigger deployment          │  │
│  │  WS   /api/ws/logs   — stream deployment logs      │  │
│  │  GET  /api/branches  — list branches for a repo    │  │
│  │  GET  /api/status    — health / prerequisites      │  │
│  └────────────────────────────────────────────────────┘  │
│                                                          │
│  Local Cloned Repos:                                     │
│    ~/.deploy_tool/repos/nbn-daemon/                      │
│    ~/.deploy_tool/repos/unity/                           │
└──────────────────────┬───────────────────────────────────┘
                       │ subprocess (make / python / ssh)
┌──────────────────────▼───────────────────────────────────┐
│           Target Filer (IP provided by user)              │
└──────────────────────────────────────────────────────────┘
```

## 5. Functional Requirements

### FR-1: Initial Repository Setup (First-Time / Server Bootstrap)

**Description**: When the deploy tool is first deployed on a machine, it must clone both repositories into a well-known, constant location so that subsequent deployments can operate on local copies.

| # | Requirement | Details |
|---|------------|--------|
| FR-1.1 | Clone repos on first run | On startup, if repos are not already cloned at the configured path, clone them using HTTPS + GitHub PAT. |
| FR-1.2 | Constant clone location | Repos are cloned to user-home relative path (default: `~/.deploy_tool/repos/<repo-name>/`). |
| FR-1.3 | GitHub authentication | Both repos use HTTPS + GitHub Personal Access Token (PAT). The PAT is provided via environment variable (`GITHUB_PAT`). Clone URL format: `https://<PAT>@github.com/nasuni/<repo>.git` |
| FR-1.4 | Startup validation | On server startup, verify that repos exist and are valid git repositories. If not, attempt to clone. Log errors clearly if cloning fails. |

### FR-2: Branch Preparation (Pre-Deployment Git Operations)

**Description**: Before each deployment, the tool must ensure the local repo is on the correct branch and up-to-date with the remote.

| # | Requirement | Details |
|---|------------|--------|
| FR-2.1 | Fetch latest from remote | Run `git fetch origin` to get all latest remote refs. |
| FR-2.2 | Checkout requested branch | Switch to the branch specified by the user. If the branch exists locally, check it out. If not, create a tracking branch from `origin/<branch>`. |
| FR-2.3 | Rebase with latest | Run `git rebase origin/<branch>` to pull in latest changes from the remote branch, keeping the local copy current. |
| FR-2.4 | Handle dirty working tree | If the working tree is dirty (uncommitted changes from a previous failed deploy), run `git reset --hard origin/<branch>` and `git clean -fd` to ensure a clean state. |
| FR-2.5 | Branch validation | Return a clear error if the specified branch does not exist on the remote. |

### FR-3: Deployment Execution

**Description**: Trigger the actual deployment using the repo's existing scripts/tooling.

| # | Requirement | Details |
|---|------------|--------|
| FR-3.1 | nbn-daemon deployment | Execute `make deploy-rpm FILER=<filerIP>` from the nbn-daemon repo root. This internally handles building the RPM (via Docker) and deploying it to the filer. |
| FR-3.2 | Unity deployment — sync | Execute `python python/tools/sync-dev.py --restart <filerIP>` from the Unity repo root. |
| FR-3.3 | Unity deployment — post-sync | After sync-dev.py completes successfully, execute via SSH (port 222): delete stale `.pyc`/`.pyo` files, then restart `filer-route-http`, `qman`, and `httpd`. |
| FR-3.4 | Subprocess execution | Run deploy commands as subprocesses, capturing both stdout and stderr in real-time. |
| FR-3.5 | Concurrent deployments | Deployments to the same repo are queued (git working directory is not concurrent-safe). Deployments to **different repos** (including to the same filer) are allowed concurrently — no cross-repo dependency. |
| FR-3.6 | Timeout handling | Deployments should have a configurable timeout (default: 30 minutes). If exceeded, kill the subprocess and report failure. |
| FR-3.7 | Exit code reporting | Capture the exit code of the deploy command. Report success (exit 0) or failure (non-zero) clearly in the UI. |

### FR-4: Real-Time Log Streaming

**Description**: The UI must display deployment logs as they are produced, not after the deployment completes.

| # | Requirement | Details |
|---|------------|--------|
| FR-4.1 | WebSocket log channel | Establish a WebSocket connection between the React frontend and FastAPI backend to stream logs in real-time. |
| FR-4.2 | Line-by-line streaming | Each line of stdout/stderr from the deployment subprocess is sent to the frontend as it is produced. |
| FR-4.3 | Log differentiation | Distinguish between stdout and stderr in the UI (e.g., stderr in red/orange). |
| FR-4.4 | Auto-scroll | The log viewer auto-scrolls to the latest output. User can scroll up to review history; auto-scroll resumes when scrolled to bottom. |
| FR-4.5 | Log persistence | Logs for the current deployment session are kept in memory. They are available as long as the WebSocket is connected. (No persistent log storage required in v1.) |

### FR-5: Bookmarkable URLs

**Description**: The user must be able to bookmark a deployment configuration so they can quickly re-trigger it.

| # | Requirement | Details |
|---|------------|--------|
| FR-5.1 | URL query parameters | The frontend URL encodes the deployment parameters as query params: `?repo=nbn-daemon&branch=feature/my-branch&filerIP=10.0.0.100` |
| FR-5.2 | Pre-fill from URL | When the page loads with query parameters, the form fields (repo, branch, filer IP) are pre-filled from the URL. |
| FR-5.3 | URL updates on input | As the user changes form fields, the URL query parameters update in real-time (using `replaceState`, no page reload). |
| FR-5.4 | One-click deploy | A user with a bookmarked URL lands on the page with all fields pre-filled. They just click "Deploy". |

### FR-6: Branch Listing & Selection

**Description**: Provide a searchable dropdown for users to select from available branches.

| # | Requirement | Details |
|---|------------|--------|
| FR-6.1 | List remote branches | Provide an API endpoint that returns the list of remote branches for a given repo (`git branch -r`). |
| FR-6.2 | Custom searchable dropdown | The branch selector is a **custom dropdown component** with built-in search/filter. Not a free-text input — user must select from existing remote branches. |
| FR-6.3 | Search/filter behavior | As the user types, the dropdown filters branches matching the typed text (case-insensitive substring match). |
| FR-6.4 | Refresh branches | A refresh button fetches latest branches from remote before populating the dropdown. |

## 6. API Specification

### 6.1 REST Endpoints

| Method | Path | Description | Request Body / Params | Response |
|--------|------|-------------|----------------------|----------|
| `GET` | `/api/status` | Health check + prerequisite validation | — | `{ "status": "ok", "repos": { "nbn-daemon": true, "unity": true }, "docker": true }` |
| `GET` | `/api/branches/{repo}` | List remote branches for a repo | `repo`: `nbn-daemon` or `unity` | `{ "branches": ["main", "feature/x", ...] }` |
| `POST` | `/api/deploy` | Trigger a deployment | `{ "repo": "nbn-daemon", "branch": "feature/x", "filerIP": "10.0.0.100" }` | `{ "deploymentId": "uuid", "status": "started" }` |
| `GET` | `/api/deployments/{id}` | Get deployment status | `id`: deployment UUID | `{ "status": "running"|"success"|"failed", "exitCode": 0 }` |

### 6.2 WebSocket Endpoint

| Path | Description | Messages |
|------|-------------|----------|
| `WS /api/ws/logs/{deploymentId}` | Stream real-time logs for a deployment | `{ "type": "stdout"|"stderr", "line": "...", "timestamp": "..." }` |

## 7. Frontend Requirements

### 7.1 UI Components

| Component | Description |
|-----------|-------------|
| **Repo Selector** | Dropdown or toggle to choose between `nbn-daemon` and `unity`. |
| **Branch Dropdown** | Custom searchable dropdown component. Populated from `/api/branches/{repo}`. Filters as user types. Only allows selection of existing remote branches. |
| **Filer IP Input** | Text input for the target filer IP address. Basic IPv4 format validation. |
| **Deploy Button** | Triggers `POST /api/deploy`. Disabled while a deployment is in progress for the same repo. Shows loading state. |
| **Log Viewer** | Terminal-like panel that displays real-time logs via WebSocket. Supports auto-scroll and manual scroll-back. Differentiates stdout (white/default) from stderr (red/orange). Styled with Tailwind. |
| **Status Indicator** | Shows current deployment status: Idle → Running → Success / Failed. |

### 7.2 URL Behavior

- URL format: `http://<host>:<port>/?repo=nbn-daemon&branch=feature/x&filerIP=10.0.0.100`
- All three query params are optional; form defaults to empty/first-available if not provided.
- Changing any input field updates the URL via `history.replaceState()`.
- On page load, query params pre-fill the corresponding form fields.
- Branch from URL is pre-selected in the dropdown (even before branch list loads — shown as selected, validated once list arrives).

### 7.3 Styling

- **Tailwind CSS** for all styling — no separate CSS files.
- Clean, minimal developer-tool aesthetic.
- Dark theme for the log viewer (terminal-like).
- Responsive layout but primarily designed for desktop use.

## 8. Prerequisites & Environment

The following must be available on the machine where the Deploy Tool server runs:

| Prerequisite | Required For | Notes |
|-------------|-------------|-------|
| **Git** | Cloning and managing repos | Must be in PATH |
| **Docker** | nbn-daemon `make deploy-rpm` (builds RPM) | Docker daemon must be running |
| **Python 3.x** | FastAPI backend, Unity sync-dev.py | |
| **uv** | nbn-daemon build/deploy scripts | Python package manager used by nbn-daemon |
| **Node.js / npm** | React frontend build | |
| **GitHub PAT** | Cloning private repos (both via HTTPS) | Provided via `GITHUB_PAT` env var |
| **SSH key** | sync-dev.py (SSH to filers, port 222) + post-sync service restarts | SSH agent must have the key loaded |
| **make** | nbn-daemon deploy | |

## 9. Configuration

The following should be configurable via environment variables or a `.env` file:

| Variable | Default | Description |
|----------|---------|-------------|
| `GITHUB_PAT` | (required) | GitHub Personal Access Token for cloning both repos via HTTPS |
| `REPOS_BASE_PATH` | `~/.deploy_tool/repos` | Directory where repos are cloned (user-home relative) |
| `NBN_DAEMON_REPO_URL` | `https://github.com/nasuni/nbn-daemon.git` | nbn-daemon repo URL |
| `UNITY_REPO_URL` | `https://github.com/nasuni/unity.git` | Unity repo URL |
| `DEPLOY_TIMEOUT_SECONDS` | `1800` (30 min) | Max time for a deployment before timeout |
| `SERVER_PORT` | `8000` | FastAPI server port |
| `FRONTEND_PORT` | `3000` | React dev server port (dev mode only) |

## 10. Deployment Flow — Step by Step

### 10.1 nbn-daemon Flow

```
User clicks "Deploy" (repo=nbn-daemon, branch=X, filerIP=Y)
       │
       ▼
┌─────────────────────────┐
│  1. Acquire repo lock   │  (prevent concurrent git ops on nbn-daemon)
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  2. git fetch origin    │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  3. Validate branch     │  (does origin/<branch> exist?)
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  4. git reset --hard    │  (clean any leftover state)
│     git clean -fd       │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  5. git checkout branch │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  6. git rebase          │
│     origin/<branch>     │
└───────────┬─────────────┘
            │
            ▼
┌───────────────────────────────────────┐
│  7. make deploy-rpm FILER=<ip>        │
│     (internally: build RPM via Docker │
│      + SCP to filer + install)        │
└───────────┬───────────────────────────┘
            │  stdout/stderr streamed via WebSocket
            ▼
┌─────────────────────────┐
│  8. Report result       │
└───────────┬─────────────┘
            ▼
┌─────────────────────────┐
│  9. Release repo lock   │
└─────────────────────────┘
```

### 10.2 Unity Flow

```
User clicks "Deploy" (repo=unity, branch=X, filerIP=Y)
       │
       ▼
┌─────────────────────────┐
│  1. Acquire repo lock   │  (prevent concurrent git ops on unity)
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  2. git fetch origin    │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  3. Validate branch     │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  4. git reset --hard    │
│     git clean -fd       │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  5. git checkout branch │
└───────────┬─────────────┘
            │
            ▼
┌─────────────────────────┐
│  6. git rebase          │
│     origin/<branch>     │
└───────────┬─────────────┘
            │
            ▼
┌───────────────────────────────────────────┐
│  7. python python/tools/sync-dev.py       │
│     --restart <filerIP>                   │
└───────────┬───────────────────────────────┘
            │  stdout/stderr streamed via WebSocket
            ▼
┌───────────────────────────────────────────┐
│  8. Post-sync (via SSH port 222):         │
│     - Delete .pyc/.pyo bytecode           │
│     - Restart filer-route-http            │
│     - Restart qman                        │
│     - Restart httpd                       │
└───────────┬───────────────────────────────┘
            │
            ▼
┌─────────────────────────┐
│  9. Report result       │
└───────────┬─────────────┘
            ▼
┌─────────────────────────┐
│  10. Release repo lock  │
└─────────────────────────┘
```

## 11. Concurrency & Locking

| Concern | Strategy |
|---------|----------|
| Same repo, concurrent deploys | Use an `asyncio.Lock` per repo. Only one deployment per repo at a time. Subsequent requests are queued. |
| Different repos, concurrent deploys | Allowed — nbn-daemon and unity can deploy simultaneously (even to the same filer) since they use separate working directories and there is no cross-repo dependency. |
| Same filer, different repos | Allowed concurrently — no dependency between nbn-daemon RPM install and Unity Python sync. |
| UI feedback for queued deploys | If a deploy is queued behind another, the UI shows "Waiting for previous deployment to finish..." with the position in queue. |

## 12. Error Handling

| Scenario | Behavior |
|----------|----------|
| Branch does not exist | Return 400 with message: "Branch '<name>' not found on remote." |
| Git fetch fails (network) | Return 502 with message: "Failed to fetch from remote. Check network/credentials." |
| Docker not running (nbn-daemon) | Return 503 with message: "Docker is not running. Required for nbn-daemon builds." |
| Deploy script exits non-zero | Mark deployment as failed, stream the error output, report exit code. |
| Post-sync SSH fails (Unity) | Mark deployment as failed with message: "Post-sync service restart failed." Stream SSH error output. |
| Deployment timeout | Kill subprocess, mark as failed with message: "Deployment timed out after {timeout}s." |
| Invalid filer IP format | Return 400 with message: "Invalid IP address format." |
| Repo not cloned / missing | Attempt to re-clone. If that fails, return 503. |

## 13. Security Considerations

| Concern | Mitigation |
|---------|------------|
| GitHub PAT exposure | PAT is stored server-side only (env var). Never sent to the frontend. Used only in git clone/fetch URLs. |
| Command injection via filer IP | Validate filer IP is a valid IPv4 address (regex + `ipaddress.ip_address()` parsing). Never interpolate raw user input into shell commands — use argument lists. |
| Command injection via branch name | Validate branch name against git ref format (`git check-ref-format`). Reject names with shell metacharacters. |
| SSH access | SSH key must be pre-configured on the host. The tool uses SSH port 222 for filer access. No credentials stored in the app beyond the SSH agent. |
| Unauthorized access | v1: The tool is intended for internal use on a dev network. No auth required initially. Future: add SSO/OAuth. |

## 14. Out of Scope (v1)

The following are explicitly **not** in scope for the initial version:

- Deployment history / audit log persistence (database)
- User authentication and authorization
- Rollback functionality
- Scheduled / automated deployments
- Multi-filer batch deployments from one trigger
- Notifications (Slack, email, etc.)
- Container-based deployment of the tool itself
- Selective service restarts based on changed files (always full restart in v1)
- NMC-specific deployments (only filer targets in v1)

## 15. Resolved Design Decisions

| # | Question | Decision |
|---|----------|----------|
| 1 | Repo clone protocol | **HTTPS + PAT for both repos.** Unity URL changed from SSH to `https://github.com/nasuni/unity.git`. |
| 2 | nbn-daemon build step | **Skip separate `make build-rpm`.** The `deploy-rpm` target internally calls `build_rpm()` which runs the Docker build. Single command: `make deploy-rpm FILER=<ip>`. |
| 3 | Unity deploy options | **`sync-dev.py --restart` + full post-sync restart.** Always delete bytecode and restart `filer-route-http` + `qman` + `httpd` after sync. No selective options in v1. |
| 4 | Concurrent deploys to same filer | **Allowed.** No cross-repo dependency between nbn-daemon and Unity deployments. |
| 5 | Repo clone location | **User-home relative: `~/.deploy_tool/repos/`** |
| 6 | Branch selection UI | **Custom searchable dropdown component.** User must select from existing remote branches (not free-text). Dropdown filters as user types. |

---

**End of Functional Requirements Document v2**  
All open questions from v1 have been resolved. Ready for implementation planning.